# RAG Evolution Lab — Dataset Preparation

This notebook downloads, inspects, and validates all benchmark datasets used across the 9 RAG stages.
Run this once before any stage eval. Datasets are cached by HuggingFace and reused automatically.

## Benchmark Assignment per Stage

| Benchmark | Dataset ID | Purpose | Stages |
|-----------|-----------|---------|--------|
| **SciFact** | `BeIR/scifact` | Open-corpus Recall@K (5,183 docs, ~1 relevant/query) | S0–S3 |
| **RAGBench TechQA** | `rungalileo/ragbench` | Closed-corpus ranking MRR/nDCG (5 passages/query) | S0–S3 |
| **MultiHopRAG** | `yixuantt/MultiHopRAG` | Multi-hop reasoning (609 articles, evidence chains) | S3, S4, S8 |
| **FinDER** | `Linq-AI-Research/FinDER` | Financial QA on SEC 10-K filings | S5, S6 |
| **FinanceBench** | `PatronusAI/financebench` | FinanceBench accuracy (supplementary to FinDER) | S5, S6 |
| **arXiv** | `arxiv-community/arxiv_dataset` | 1.7M paper abstracts (main corpus S0–S4) | S0–S4, S7, S8 |
| **SEC 10-K** | `PleIAs/SEC` | 10K filing subset for vectorless RAG | S5, S6, S7 |

## Why Two Benchmarks for S0–S2?

RAGBench gives only 5 pre-selected passages per question → Recall@10 is permanently 0.80 regardless of method.  
SciFact has 5,183 documents → retrieval methods genuinely differ. Both are needed:
- **SciFact**: measures *can the system find the right doc from 5K?*  
- **RAGBench**: measures *does the system rank the right doc highest among 5?*

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

print('datasets library ready')

## 1. SciFact (BeIR) — Open-Corpus Retrieval Benchmark

In [ ]:
# Load SciFact corpus, queries, and relevance judgements
print('Loading SciFact...')
scifact_corpus  = load_dataset('BeIR/scifact', 'corpus', split='corpus')
scifact_queries = load_dataset('BeIR/scifact', 'queries', split='queries')
scifact_qrels   = load_dataset('BeIR/scifact-qrels', split='test')

print(f'Corpus:  {len(scifact_corpus):,} documents')
print(f'Queries: {len(scifact_queries):,} total, {len(set(str(r["query-id"]) for r in scifact_qrels)):,} with test qrels')
print(f'Qrels:   {len(scifact_qrels):,} relevance judgements')

# Relevance distribution
q2rel = defaultdict(list)
for r in scifact_qrels:
    if r['score'] > 0:
        q2rel[r['query-id']].append(r['corpus-id'])
counts = [len(v) for v in q2rel.values()]
print(f'\nRelevant docs per query: avg={sum(counts)/len(counts):.2f}, min={min(counts)}, max={max(counts)}')
print(f'Queries with 1 relevant: {counts.count(1)} ({counts.count(1)/len(counts)*100:.0f}%)')

# Sample query
sample_q = scifact_queries[0]
print(f'\nSample query: "{sample_q["text"][:100]}"')
print(f'Sample corpus doc: "{scifact_corpus[0]["title"][:80]}"')

In [ ]:
# Visualise corpus text length distribution
import matplotlib.pyplot as plt

lengths = [len((r['title'] + ' ' + r['text']).split()) for r in scifact_corpus]
plt.figure(figsize=(9, 3))
plt.hist(lengths, bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
plt.xlabel('Document length (words)')
plt.ylabel('Count')
plt.title('SciFact: Document Length Distribution (5,183 abstracts)')
plt.axvline(sum(lengths)/len(lengths), color='red', linestyle='--', label=f'Mean={sum(lengths)/len(lengths):.0f}')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Mean length: {sum(lengths)/len(lengths):.0f} words, Max: {max(lengths)} words')

## 2. RAGBench TechQA — Closed-Corpus Ranking Benchmark

In [ ]:
print('Loading RAGBench TechQA...')
ragbench = load_dataset('rungalileo/ragbench', 'techqa', split='test')

print(f'Total queries: {len(ragbench):,}')
print(f'Passages per query: always 5 (closed-corpus)')

# Check relevant doc distribution
relevant_counts = []
empty_qrels = 0
for row in ragbench:
    rk = row.get('all_relevant_sentence_keys', [])
    rd = {k.rstrip('abcdefghijklmnopqrstuvwxyz') for k in rk}
    rd.discard('')
    if not rd:
        empty_qrels += 1
    relevant_counts.append(len(rd))

print(f'\nQueries with NO relevant doc: {empty_qrels} ({empty_qrels/len(ragbench)*100:.0f}%)')
print(f'Queries WITH relevant doc: {len(ragbench)-empty_qrels} ({(len(ragbench)-empty_qrels)/len(ragbench)*100:.0f}%)')
print(f'\n⚠ Recall@10 is permanently locked at {(len(ragbench)-empty_qrels)/len(ragbench):.2f}')
print('   because we always return all 5 passages — retrieval cannot miss anything.')
print('   Only MRR and nDCG@10 meaningfully differentiate methods on this benchmark.')

# Sample
print(f'\nSample query: "{ragbench[0]["question"][:100]}"')
print(f'Sample passages: {len(ragbench[0]["documents"])} passages provided')

## 3. MultiHopRAG — Multi-Hop Retrieval Benchmark (S3, S4, S8)

In [ ]:
print('Loading MultiHopRAG...')
multihop_queries = load_dataset('yixuantt/MultiHopRAG', 'MultiHopRAG', split='train')
multihop_corpus  = load_dataset('yixuantt/MultiHopRAG', 'corpus', split='train')

print(f'Queries: {len(multihop_queries):,}')
print(f'Corpus:  {len(multihop_corpus):,} articles')

# Evidence chain length distribution
ev_lengths = [len(row['evidence_list']) for row in multihop_queries]
from collections import Counter
print(f'\nEvidence chains per query: {dict(Counter(ev_lengths))}')
print(f'Avg evidence docs per query: {sum(ev_lengths)/len(ev_lengths):.2f}')
print(f'\nThis is why single-pass RAG struggles: {sum(1 for l in ev_lengths if l>1)} queries need 2+ hops.')

# Sample
row = multihop_queries[0]
print(f'\nSample: "{row["query"][:120]}"')
print(f'Needs {len(row["evidence_list"])} evidence documents from {len(multihop_corpus)} article corpus')

In [ ]:
# Visualise evidence chain lengths
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MultiHopRAG — Why Single-Pass RAG Fails', fontsize=12, fontweight='bold')

# Evidence chain distribution
counts = list(Counter(ev_lengths).items())
counts.sort()
axes[0].bar([str(c[0]) for c in counts], [c[1] for c in counts], color='#E91E63')
axes[0].set_xlabel('Evidence documents needed')
axes[0].set_ylabel('Number of queries')
axes[0].set_title('Evidence Chain Length Distribution')

# Why agents help: illustration
x = ['Single-pass\nRAG (S0-S2)', 'Agentic\nRAG (S3)', 'CoRAG\nIterative (S8)']
y_single = [0.45, 0.75, 0.85]  # expected pattern
axes[1].bar(x, y_single, color=['#2196F3','#E91E63','#9C27B0'], alpha=0.8)
axes[1].set_ylabel('Expected Recall (multi-hop queries)')
axes[1].set_title('Expected Pattern: Multi-Hop Accuracy')
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(y_single):
    axes[1].text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Current Measured Results (Auto-loaded from results/)

In [ ]:
import json
from pathlib import Path

def load_best(stage_dir, benchmark=None):
    files = sorted(stage_dir.glob('*.json'))
    if benchmark:
        matches = [f for f in files if json.loads(f.read_text()).get('benchmark') == benchmark]
        files = matches if matches else files
    for f in reversed(files):
        d = json.loads(f.read_text())
        if d.get('recall_at_10', 0) > 0 or d.get('mrr', 0) > 0:
            return d
    return {}

# SciFact results
print('=== SciFact Open-Corpus Results (50 queries, 5,183 docs) ===')
sc_results = {}
for sn, label in [(0,'S0 BM25'), (1,'S1 Naive RAG'), (2,'S2 Advanced')]:
    sd = Path(f'results/stage_{sn}')
    d = load_best(sd, benchmark='scifact')
    if d:
        sc_results[label] = d
        print(f'  {label:<20} R@10={d["recall_at_10"]:.4f}  MRR={d["mrr"]:.4f}  nDCG={d["ndcg_at_10"]:.4f}  p99={d["latency_p99_ms"]/1000:.0f}s')

print()
print('=== RAGBench Ranking Results (50 queries, 5 passages/query) ===')
for sn, label in [(0,'S0 BM25'), (1,'S1 Naive RAG'), (2,'S2 Advanced')]:
    sd = Path(f'results/stage_{sn}')
    d = load_best(sd, benchmark='ragbench')
    if d:
        print(f'  {label:<20} R@10={d["recall_at_10"]:.4f}(locked)  MRR={d["mrr"]:.4f}  nDCG={d["ndcg_at_10"]:.4f}')

print()
print('=== MultiHopRAG Results (S3 Agentic) ===')
d3 = load_best(Path('results/stage_3'), benchmark='multihop')
if d3:
    print(f'  S3 Agentic (N={d3["n_queries"]}): R@10={d3["recall_at_10"]:.4f}  MRR={d3["mrr"]:.4f}  nDCG={d3["ndcg_at_10"]:.4f}  p99={d3["latency_p99_ms"]/1000:.0f}s')
    print('  (5-query sample; increase n_samples for statistically robust numbers)')

In [ ]:
# Summary comparison chart — SciFact (the meaningful Recall@K data)
if sc_results:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('SciFact Open-Corpus: Stage Progression\n(5,183 docs, ~1 relevant/query — the benchmark where method choice matters)',
                 fontsize=11, fontweight='bold')
    names = list(sc_results.keys())
    colors = ['#2196F3', '#4CAF50', '#FF9800'][:len(names)]

    for ax, (metric, title) in zip(axes, [('recall_at_10','Recall@10'),('mrr','MRR'),('ndcg_at_10','nDCG@10')]):
        vals = [sc_results[n][metric] for n in names]
        bars = ax.bar(range(len(names)), vals, color=colors, width=0.5, zorder=3)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels([n.replace(' ',chr(10)) for n in names], fontsize=8)
        ax.set_ylim(0.5, 1.0)
        ax.set_title(title, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, zorder=0)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.4f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        for i in range(1, len(names)):
            d = vals[i]-vals[i-1]
            clr = '#2e7d32' if d>0 else '#c62828'
            ax.text(i, vals[i]+0.025, f'+{d:.4f}' if d>=0 else f'{d:.4f}',
                    ha='center', va='bottom', fontsize=7.5, color=clr, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Pending Datasets (Stages 5–8)

These will be downloaded when the corresponding stages are implemented.

In [ ]:
# Preview dataset metadata without downloading full content
print('Pending dataset sizes (will be downloaded at stage build time):')
pending = [
    ('FinDER',       'Linq-AI-Research/FinDER',       '5,703 expert-labeled 10-K QA triplets',      'S5, S6'),
    ('FinanceBench', 'PatronusAI/financebench',        '150 financial QA samples',                   'S5, S6'),
    ('arXiv',        'arxiv-community/arxiv_dataset',  '1.7M paper abstracts, ~3GB',                 'S0-S4, S7, S8'),
    ('SEC 10-K',     'PleIAs/SEC',                     '245K filings (use 10K subset), ~6GB subset', 'S5, S6, S7'),
]
print(f'{"Dataset":<15} {"HF ID":<40} {"Size/Notes":<45} {"Stages"}')
print('-'*120)
for name, hf_id, notes, stages in pending:
    print(f'{name:<15} {hf_id:<40} {notes:<45} {stages}')

print()
print('To download arXiv (takes ~2h, ~3GB):')
print('  uv run python scripts/download_arxiv.py --output data/arxiv_abstracts.json')
print('  (use --n-samples 10000 for quick iteration)')